# Detector de barra — Fase 1 + Fase 2 en un solo notebook de Kaggle

Reemplaza el flujo Colab (Fase 1) + Kaggle (Fase 2): todo corre acá, sin descargar/resubir nada
entre medio. Pensado para "Save & Run All" (Commit) — corre server-side en Kaggle aunque cierres
la pestaña, a diferencia de Colab free que se desconecta y pierde todo si cerrás la sesión.

## Antes de correr (una sola vez)

1. **Settings del notebook** (panel derecho): Accelerator = **GPU** (P100 o T4x2), Internet = **ON**
   (hace falta para Roboflow y Hugging Face).
2. **Add-ons > Secrets**: agregá un secreto llamado `ROBOFLOW_API_KEY` con tu API key gratis de
   Roboflow (se crea en https://app.roboflow.com/settings/api). Esto reemplaza al `getpass()` que
   usábamos en Colab — en "Save & Run All" no hay nadie tipeando, así que no puede depender de un
   input interactivo.
3. Click en **Save & Run All (Commit)**. No hace falta quedarse mirando — volvé en ~6-7h y
   descargá `resultado_entrenamiento.zip` del tab **Output** del notebook ya commiteado.

In [ ]:
from kaggle_secrets import UserSecretsClient
ROBOFLOW_API_KEY = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")


## Fase 1 — Armar el dataset combinado

Ver `README.md` de esta carpeta para la tabla completa de clases, fuentes y licencias (todas CC
BY 4.0). Mismo contenido que `01_assemble_dataset.ipynb`, con dos diferencias para que corra
desatendido: la API key viene de Secrets (celda anterior) en vez de `getpass()`, y no hay
descarga al navegador — el zip queda en `/kaggle/working/` para el tab Output.

In [ ]:
!pip install -q roboflow autodistill autodistill-grounding-dino supervision
# Pin de versiones compatibles (ver notas en 01_assemble_dataset.ipynb sobre por que hace falta):
# transformers/tokenizers anteriores a que HF le saque metodos que el GroundingDINO viejo usa,
# numpy/scipy como par compatible conocido, todo en un solo comando para que pip resuelva junto.
import numpy as np
!pip install -q "transformers==4.49.0" "tokenizers==0.21.0" "numpy==2.1.3" "scipy==1.14.1"
import importlib.metadata as md, os
if np.__version__ != md.version("numpy"):
    print("numpy cambió en disco: hace falta reiniciar el kernel. En un commit de 'Save & Run All' "
          "esto no debería pasar (kernel nuevo, sin numpy cargado todavía) — si pasa corriendo a mano, "
          "Kernel > Restart y correr de nuevo desde la celda de Secrets.")


In [ ]:
from roboflow import Roboflow
import os

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
RAW_DIR = "/kaggle/working/raw_datasets"
os.makedirs(RAW_DIR, exist_ok=True)

SOURCES = [
    # (workspace, project, version, carpeta_local)
    ("computer-vision-1bdzc", "barbell-detection-yolov5", 6, "barbell_a"),
    ("nkust-airs-lps", "nkust_barbell", 3, "barbell_b"),
    ("ecnu", "gym_001", 1, "gym_001"),
    ("gymitemidentifier", "gym_greyscale", 1, "gym_greyscale"),
]

downloaded = []
for workspace, project, version, folder in SOURCES:
    try:
        ds = rf.workspace(workspace).project(project).version(version).download(
            "yolov8", location=os.path.join(RAW_DIR, folder)
        )
        downloaded.append(folder)
        print(f"OK: {project} -> {ds.location}")
    except Exception as e:
        print(f"FALLÓ {project} (revisá que la versión {version} siga existiendo en Roboflow): {e}")

print("\nDescargados:", downloaded)


In [ ]:
import glob, shutil, yaml

MASTER_CLASSES = ["barbell", "dumbbell", "kettlebell", "plate", "cable"]
CLASS_ALIASES = {"barbell": "barbell", "Barbell": "barbell"}  # normaliza mayúscula distinta

MERGED_DIR = "/kaggle/working/merged_public"
for split in ["train", "valid", "test"]:
    os.makedirs(f"{MERGED_DIR}/{split}/images", exist_ok=True)
    os.makedirs(f"{MERGED_DIR}/{split}/labels", exist_ok=True)

def remap_and_copy(src_folder, src_classes):
    """Copia imágenes/labels de un dataset descargado, remapeando ids de clase al índice de MASTER_CLASSES.
    Se salta (no copia) cualquier label de una clase que no esté en CLASS_ALIASES/MASTER_CLASSES."""
    id_map = {}
    for i, name in enumerate(src_classes):
        canon = CLASS_ALIASES.get(name, name)
        if canon in MASTER_CLASSES:
            id_map[i] = MASTER_CLASSES.index(canon)
    n_copied = 0
    for split_src, split_dst in [("train", "train"), ("valid", "valid"), ("test", "test")]:
        img_dir = os.path.join(src_folder, split_src, "images")
        lbl_dir = os.path.join(src_folder, split_src, "labels")
        if not os.path.isdir(img_dir):
            continue
        for img_path in glob.glob(os.path.join(img_dir, "*")):
            stem = os.path.splitext(os.path.basename(img_path))[0]
            lbl_path = os.path.join(lbl_dir, stem + ".txt")
            new_lines = []
            if os.path.exists(lbl_path):
                for line in open(lbl_path):
                    parts = line.split()
                    if not parts:
                        continue
                    cid = int(parts[0])
                    if cid in id_map:
                        new_lines.append(f"{id_map[cid]} {' '.join(parts[1:])}")
            if not new_lines:
                continue
            tag = os.path.basename(src_folder)
            out_img = os.path.join(MERGED_DIR, split_dst, "images", f"{tag}_{stem}{os.path.splitext(img_path)[1]}")
            out_lbl = os.path.join(MERGED_DIR, split_dst, "labels", f"{tag}_{stem}.txt")
            shutil.copy(img_path, out_img)
            open(out_lbl, "w").write("\n".join(new_lines))
            n_copied += 1
    return n_copied

for folder in downloaded:
    data_yaml = yaml.safe_load(open(os.path.join(RAW_DIR, folder, "data.yaml")))
    n = remap_and_copy(os.path.join(RAW_DIR, folder), data_yaml["names"])
    print(f"{folder}: {n} imágenes copiadas al dataset fusionado")


In [ ]:
from autodistill_grounding_dino import GroundingDINO
from autodistill.detection import CaptionOntology

BOX_THRESHOLD = 0.35

NEW_CLASSES_ONTOLOGY = CaptionOntology({
    "straight olympic weightlifting barbell with knurled grip, long metal bar used for squats or bench press": "barra_olimpica",
    "hexagonal trap bar for deadlifts, hexagon shaped barbell with handles on both sides": "barra_hexagonal",
    "short EZ curl bar with zigzag shape for bicep curls": "barra_curl",
    "cable machine tower with pulley and weight stack in a gym": "maquina_polea",
})

base_model = GroundingDINO(ontology=NEW_CLASSES_ONTOLOGY, box_threshold=BOX_THRESHOLD)

UNLABELED_POOL = "/kaggle/working/autodistill_pool"
os.makedirs(UNLABELED_POOL, exist_ok=True)
for folder in downloaded:
    for split in ["train", "valid", "test"]:
        img_dir = os.path.join(RAW_DIR, folder, split, "images")
        if not os.path.isdir(img_dir):
            continue
        for img_path in glob.glob(os.path.join(img_dir, "*")):
            dst = os.path.join(UNLABELED_POOL, f"{folder}_{os.path.basename(img_path)}")
            if not os.path.exists(dst):
                shutil.copy(img_path, dst)

print(f"Imágenes en el pool para Autodistill: {len(os.listdir(UNLABELED_POOL))}")

AUTODISTILL_OUT = "/kaggle/working/autodistill_labeled"
base_model.label(input_folder=UNLABELED_POOL, output_folder=AUTODISTILL_OUT)


In [ ]:
# Muestra de lo que etiquetó Autodistill — queda en el Output del commit para revisar después,
# no bloquea nada (no es un display interactivo que espere input).
import random
from PIL import Image

sample_imgs = random.sample(
    glob.glob(os.path.join(AUTODISTILL_OUT, "train", "images", "*")),
    k=min(12, len(glob.glob(os.path.join(AUTODISTILL_OUT, "train", "images", "*")))),
)
for p in sample_imgs:
    print(os.path.basename(p))
    display(Image.open(p).resize((320, 240)))


In [ ]:
FINAL_CLASSES = MASTER_CLASSES + ["barra_olimpica", "barra_hexagonal", "barra_curl", "maquina_polea"]
FINAL_DIR = "/kaggle/working/final_dataset"

for split in ["train", "valid", "test"]:
    os.makedirs(f"{FINAL_DIR}/{split}/images", exist_ok=True)
    os.makedirs(f"{FINAL_DIR}/{split}/labels", exist_ok=True)
    for img_path in glob.glob(os.path.join(MERGED_DIR, split, "images", "*")):
        shutil.copy(img_path, f"{FINAL_DIR}/{split}/images/")
    for lbl_path in glob.glob(os.path.join(MERGED_DIR, split, "labels", "*")):
        shutil.copy(lbl_path, f"{FINAL_DIR}/{split}/labels/")

offset = len(MASTER_CLASSES)
ad_images = glob.glob(os.path.join(AUTODISTILL_OUT, "train", "images", "*"))
random.seed(42)
random.shuffle(ad_images)
n = len(ad_images)
splits = {
    "train": ad_images[: int(n * 0.7)],
    "valid": ad_images[int(n * 0.7): int(n * 0.9)],
    "test": ad_images[int(n * 0.9):],
}
for split, imgs in splits.items():
    for img_path in imgs:
        stem = os.path.splitext(os.path.basename(img_path))[0]
        src_lbl = os.path.join(AUTODISTILL_OUT, "train", "labels", stem + ".txt")
        if not os.path.exists(src_lbl):
            continue
        new_lines = []
        for line in open(src_lbl):
            parts = line.split()
            if not parts:
                continue
            cid = int(parts[0]) + offset
            new_lines.append(f"{cid} {' '.join(parts[1:])}")
        shutil.copy(img_path, f"{FINAL_DIR}/{split}/images/ad_{stem}{os.path.splitext(img_path)[1]}")
        open(f"{FINAL_DIR}/{split}/labels/ad_{stem}.txt", "w").write("\n".join(new_lines))

data_yaml_path = os.path.join(FINAL_DIR, "data.yaml")
yaml.safe_dump(
    {"train": "train/images", "val": "valid/images", "test": "test/images",
     "nc": len(FINAL_CLASSES), "names": FINAL_CLASSES},
    open(data_yaml_path, "w"),
)

print("Clases finales:", FINAL_CLASSES)
for split in ["train", "valid", "test"]:
    n_imgs = len(glob.glob(f"{FINAL_DIR}/{split}/images/*"))
    print(f"{split}: {n_imgs} imágenes")


## Fase 2 — Entrenar YOLOv8n sobre el dataset recién armado

`epochs=300` + `patience=50` (early stopping): corta solo si no mejora en 50 épocas. YOLOv8n
("nano") a propósito, porque la idea es exportarlo al celular después (Fase 4).

In [ ]:
!pip install -q ultralytics


In [ ]:
from ultralytics import YOLO
import time

model = YOLO("yolov8n.pt")

start = time.time()
results = model.train(
    data=data_yaml_path,
    epochs=300,
    patience=50,
    imgsz=640,
    batch=-1,
    device=0,
    project="/kaggle/working/runs",
    name="barbell_detector",
)
elapsed_hours = (time.time() - start) / 3600
print(f"Entrenamiento terminado en {elapsed_hours:.2f} horas")


## Fase 3 — Resultados reales sobre el set de validación (nunca sobre training)

In [ ]:
best_model = YOLO("/kaggle/working/runs/barbell_detector/weights/best.pt")
metrics = best_model.val(data=data_yaml_path, split="val")

print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print("\nPor clase (mAP50):")
for i, name in metrics.names.items():
    print(f"  {name}: {metrics.box.ap50[i]:.4f}")


In [ ]:
test_images = glob.glob(os.path.join(FINAL_DIR, "test", "images", "*"))
sample = random.sample(test_images, k=min(10, len(test_images)))
pred_results = best_model.predict(sample, conf=0.25, save=True, project="/kaggle/working/runs", name="test_preview")

for r in pred_results:
    print(os.path.basename(r.path), "->", len(r.boxes), "detecciones")
    display(Image.fromarray(r.plot()[:, :, ::-1]).resize((480, 360)))


## Fase 4 (solo investigación) — exportar a ONNX, medir peso real para navegador

In [ ]:
onnx_path = best_model.export(format="onnx", imgsz=640)
size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
print(f"ONNX exportado: {onnx_path} — {size_mb:.1f} MB")


## Empaquetar todo para descargar desde el tab Output

In [ ]:
OUT_DIR = "/kaggle/working/resultado_entrenamiento"
os.makedirs(OUT_DIR, exist_ok=True)
shutil.copy("/kaggle/working/runs/barbell_detector/weights/best.pt", OUT_DIR)
shutil.copy(onnx_path, OUT_DIR)
for plot in glob.glob("/kaggle/working/runs/barbell_detector/*.png"):
    shutil.copy(plot, OUT_DIR)
shutil.copytree("/kaggle/working/runs/test_preview", os.path.join(OUT_DIR, "test_preview"), dirs_exist_ok=True)

with open(os.path.join(OUT_DIR, "resumen.txt"), "w") as f:
    f.write(f"mAP50: {metrics.box.map50:.4f}\n")
    f.write(f"mAP50-95: {metrics.box.map:.4f}\n")
    f.write(f"Horas de entrenamiento: {elapsed_hours:.2f}\n")
    f.write(f"Tamaño ONNX: {size_mb:.1f} MB\n")

shutil.make_archive("/kaggle/working/resultado_entrenamiento", "zip", OUT_DIR)
print("Listo: /kaggle/working/resultado_entrenamiento.zip — aparece en el tab Output del commit.")
